# 07 · Databricks SQL: el SQL warehouse

⏱ **12 min** · Módulo 6 · Slide de la Intro: 23 (Databricks SQL: analítica)

**Objetivo:** conocer el SQL warehouse, el cómputo de los analistas, y consultar desde él las mismas tablas que construyó el pipeline.

```
Analistas y herramientas de BI ──► SQL warehouse ──► Tablas Delta con Unity Catalog
Editor SQL · Dashboards · Alertas    se enciende, escala      gold_network_daily
                                     y se apaga solo
```

| Paso | Qué haces |
|---|---|
| 1 | Ver el SQL warehouse |
| 2 | Enviarle una consulta, como una herramienta de BI |
| 3 | Una sola fuente de verdad |
| 4 | El editor SQL: escribe, ejecuta, visualiza, comparte |
| 5 | Un dashboard y una alerta |

**Requiere:** notebook 05.

In [0]:
%run ./00_Setup

## Paso 1 · Ver el SQL warehouse

Hasta aquí todo corrió en el cómputo del notebook. Los analistas usan otro cómputo, dedicado solo a SQL.

In [0]:
warehouse = find_warehouse()

print("Nombre :", warehouse["name"])
print("Tamaño :", warehouse["cluster_size"])
print("Estado :", warehouse["state"])
print("Se apaga solo tras", warehouse["auto_stop_mins"], "minutos sin uso")

👀 **Qué observar:** es un cómputo que **se enciende** cuando llega una consulta y **se apaga** cuando nadie lo usa: pagas solo mientras trabaja.
Míralo en el menú **SQL Warehouses**. La pestaña **Connection details** tiene los datos con los que se conectan Power BI, Tableau o Excel.

## Paso 2 · Enviarle una consulta, como una herramienta de BI

`query_warehouse()` no ejecuta la consulta aquí: la envía al warehouse y espera el resultado.

In [0]:
resultado = query_warehouse("""
    SELECT network_name, SUM(episodes) AS total
    FROM gold_network_daily
    GROUP BY network_name
    ORDER BY total DESC
    LIMIT 10
""")
display(resultado)

In [0]:
print("Estado del warehouse:", find_warehouse()["state"])

👀 **Qué observar:** es la consulta del slide 24 (`SELECT region, SUM(monto) ...`) sobre nuestra tabla gold.
Si el warehouse estaba apagado (`STOPPED`), se encendió solo en unos segundos.

## Paso 3 · Una sola fuente de verdad

La misma pregunta, respondida por los dos cómputos.

In [0]:
pregunta = "SELECT SUM(episodes) AS episodios FROM gold_network_daily"

en_notebook  = spark.sql(pregunta).first()["episodios"]
en_warehouse = query_warehouse(pregunta)["episodios"][0]

print("Cómputo del notebook (ingeniería):", en_notebook)
print("SQL warehouse (analistas)        :", en_warehouse)

👀 **Qué observar:** el mismo número. No hay una copia de los datos para los analistas: ingeniería y análisis leen **la misma tabla Delta**,
gobernada por Unity Catalog. Es la promesa del lakehouse.

## Paso 4 · El editor SQL

El ciclo del slide 24: **escribe → ejecuta → visualiza → comparte o programa**. Copia el nombre de la tabla:

In [0]:
print(f"{CATALOG}.{SCHEMA}.{GOLD}")

1. **Escribe:** menú **SQL Editor**, crea una consulta nueva y pega la consulta del paso 2 con el nombre completo de la tabla.
2. **Ejecuta:** revisa que arriba esté elegido tu SQL warehouse y pulsa **Run**.
3. **Visualiza:** en el resultado, pulsa **+** y agrega una visualización de barras.
4. **Comparte o programa:** guarda la consulta. Desde ahí puedes compartirla o programarla para que corra sola.

## Paso 5 · Un dashboard y una alerta

**Dashboard** (menú **Dashboards**):

1. Crea un dashboard nuevo.
2. En la pestaña de datos, agrega un conjunto de datos con SQL: `SELECT * FROM` y el nombre completo de la tabla gold.
3. En el lienzo, agrega un gráfico de barras: eje X `airdate`, eje Y la suma de `episodes`.
4. Agrega otro con las redes (`network_name`) que tienen más episodios.
5. Publícalo.

**Alerta** (menú **Alerts**): avisa cuando un dato cruza un límite.

1. Crea una alerta con esta consulta (con el nombre completo de la tabla):
   `SELECT SUM(episodes) AS episodios FROM gold_network_daily WHERE airdate = (SELECT MAX(airdate) FROM gold_network_daily)`
2. Condición: `episodios` menor que `100`. Si el último día trae muy pocos episodios, algo falló en la ingesta.

## ✔ Checkpoint

- [ ] Sé qué es un SQL warehouse y por qué se enciende y se apaga solo
- [ ] Envié una consulta al warehouse y obtuve el mismo resultado que en el notebook
- [ ] Ejecuté y guardé una consulta en el editor SQL
- [ ] Creé un dashboard con dos gráficos y una alerta

**Siguiente:** `08_SQL_Consultas_y_Gestion_de_Datos`